In [ ]:
from crewai import LLM, Crew, Agent, Task
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import SentenceTransformerEmbeddings  # Use the wrapper


#from langchain.agents import tool
from crewai.tools import tool

In [1]:
from com.example.agentic.agent.LLMManager import LLMManager
# groq, openai
llm = LLMManager.create_llm('openai')
llm.call('Hello')

'Hello. Is there something I can help you with or would you like to chat?'

In [1]:
from langchain_community.document_loaders import UnstructuredURLLoader

# Website Data Ingestion 
loader = UnstructuredURLLoader(urls=["https://docs.crewai.com/how-to/Installing-CrewAI/"])
documents = loader.load()
print(f"Total documents {len(documents)}")
documents


Total documents 1


[Document(metadata={'source': 'https://docs.crewai.com/how-to/Installing-CrewAI/'}, page_content='Skip to main content\n\nCrewAI home page\n\nlight logo\n\ndark logo\n\nHomeDocumentationAMPAPI ReferenceExamplesChangelog\n\nWebsite\n\nForum\n\nBlog\n\nCrewGPT\n\nWelcome\n\nCrewAI Documentation\n\nWelcome\n\nCrewAI Documentation\n\nBuild collaborative AI agents, crews, and flows — production ready from day one.\n\nShip multi‑agent systems with confidence\n\nDesign agents, orchestrate crews, and automate flows with guardrails, memory, knowledge, and observability baked in.\n\nGet startedView changelogAPI Reference\n\nGet started\n\nIntroduction\n\nOverview of CrewAI concepts, architecture, and what you can build with agents, crews, and flows.\n\nInstallation\n\nInstall via uv, configure API keys, and set up the CLI for local development.\n\nQuickstart\n\nSpin up your first crew in minutes. Learn the core runtime, project layout, and dev loop.\n\nBuild the basics\n\nAgents\n\nCompose agent

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
_chunks = _splitter.split_documents(documents)
print(f"Total chunks {len(_chunks)}")
_chunks

Total chunks 3


[Document(metadata={'source': 'https://docs.crewai.com/how-to/Installing-CrewAI/'}, page_content='Skip to main content\n\nCrewAI home page\n\nlight logo\n\ndark logo\n\nHomeDocumentationAMPAPI ReferenceExamplesChangelog\n\nWebsite\n\nForum\n\nBlog\n\nCrewGPT\n\nWelcome\n\nCrewAI Documentation\n\nWelcome\n\nCrewAI Documentation\n\nBuild collaborative AI agents, crews, and flows — production ready from day one.\n\nShip multi‑agent systems with confidence\n\nDesign agents, orchestrate crews, and automate flows with guardrails, memory, knowledge, and observability baked in.\n\nGet startedView changelogAPI Reference\n\nGet started\n\nIntroduction\n\nOverview of CrewAI concepts, architecture, and what you can build with agents, crews, and flows.\n\nInstallation\n\nInstall via uv, configure API keys, and set up the CLI for local development.\n\nQuickstart\n\nSpin up your first crew in minutes. Learn the core runtime, project layout, and dev loop.\n\nBuild the basics\n\nAgents\n\nCompose agent

In [3]:
_chunk_text = [d.page_content for d in _chunks]
_chunk_text

['Skip to main content\n\nCrewAI home page\n\nlight logo\n\ndark logo\n\nHomeDocumentationAMPAPI ReferenceExamplesChangelog\n\nWebsite\n\nForum\n\nBlog\n\nCrewGPT\n\nWelcome\n\nCrewAI Documentation\n\nWelcome\n\nCrewAI Documentation\n\nBuild collaborative AI agents, crews, and flows — production ready from day one.\n\nShip multi‑agent systems with confidence\n\nDesign agents, orchestrate crews, and automate flows with guardrails, memory, knowledge, and observability baked in.\n\nGet startedView changelogAPI Reference\n\nGet started\n\nIntroduction\n\nOverview of CrewAI concepts, architecture, and what you can build with agents, crews, and flows.\n\nInstallation\n\nInstall via uv, configure API keys, and set up the CLI for local development.\n\nQuickstart\n\nSpin up your first crew in minutes. Learn the core runtime, project layout, and dev loop.\n\nBuild the basics\n\nAgents\n\nCompose agents with tools, memory, knowledge, and structured outputs using Pydantic. Includes templates and b

In [4]:
from langchain_ollama import OllamaEmbeddings
import numpy as np

# dimensions=768
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

_vectors = embeddings.embed_documents(_chunk_text)
my_array = np.array(_vectors)
print(f"embedding shape {my_array.shape}")
my_array

embedding shape (3, 768)


array([[ 0.00757645,  0.0303378 , -0.17497389, ..., -0.02236513,
        -0.07890734, -0.06454345],
       [-0.00394517, -0.02367691, -0.17507681, ..., -0.02281908,
        -0.06680782, -0.05337684],
       [-0.00664927,  0.03896915, -0.16930479, ..., -0.04778492,
        -0.05327208, -0.04809038]], shape=(3, 768))

#### This example walks through building a retrieval augmented generation (RAG) application using Ollama and embedding models.

In [ ]:
# Step 1: Generate embeddings
import os
import chromadb

## Create a simple txt file
work_dir = os.getenv("WORK_DIR")
persist_directory = f"{work_dir}/vectorstore/chroma"
client = chromadb.PersistentClient(path=persist_directory)

#
collection = client.get_or_create_collection(name="docs")

In [ ]:
# Step 1: Generate embeddings
import ollama

documents = [
  "Llamas are members of the camelid family meaning they're pretty closely related to vicuñas and camels",
  "Llamas were first domesticated and used as pack animals 4,000 to 5,000 years ago in the Peruvian highlands",
  "Llamas can grow as much as 6 feet tall though the average llama between 5 feet 6 inches and 5 feet 9 inches tall",
  "Llamas weigh between 280 and 450 pounds and can carry 25 to 30 percent of their body weight",
  "Llamas are vegetarians and have very efficient digestive systems",
  "Llamas live to be about 20 years old, though some only live for 15 years and others live to be 30 years old",
]

#client = chromadb.Client()
#collection = client.create_collection(name="docs")

# store each document in a vector embedding database
for i, d in enumerate(documents):
  # mxbai-embed-large
  response = ollama.embed(model="nomic-embed-text:latest", input=d)
  embeddings = response["embeddings"]
  collection.add(
    ids=[str(i)],
    embeddings=embeddings,
    documents=[d]
  )

In [ ]:
# Step 2: Retrieve : the most relevant document given an example prompt:

import ollama

prompt = "What animals are llamas related to ?"

# generate an embedding for the input and retrieve the most relevant doc
response = ollama.embed(
  model="nomic-embed-text:latest",
  input=prompt
)

results = collection.query(
  query_embeddings=response["embeddings"],
  n_results=1
)

data = results['documents'][0][0]

In [ ]:
data

In [ ]:
#
# Step 3: Generate : response combining the prompt and data we retrieved in step 2
#
input = "What animals are llamas related to ?"
output = ollama.generate(
  model="gemma4:latest",
  prompt=f"Using this data: {data}. Respond to this prompt: {input}"
)

print(output['response'])

In [1]:
import ollama

single = ollama.embed(
  model='nomic-embed-text:latest',
  input='The sky is blue because of Rayleigh scattering'
)
print(single['embeddings'][0])  # vector length
print(len(single['embeddings'][0]))  # vector length

[0.023892084, 0.022229966, -0.15269239, -0.010197603, 0.056493305, 0.08489726, -0.0042790417, 0.0046243374, 0.02865354, -0.06537465, 0.035416484, 0.075495146, 0.07535521, 0.07798032, 0.0173951, 0.02618654, 0.0024474177, -0.035946146, 0.0071941707, 0.015974646, -0.06760857, -0.03763745, 0.012154261, -0.056366216, 0.053398967, 0.07398132, -0.015732786, -0.013338346, -0.0020769748, -0.026457911, 0.049680237, -0.037722107, -0.020525176, -0.04244258, -0.02565084, -0.024173232, 0.05399797, -0.002309125, 0.014618152, -0.00234048, 0.022236845, -0.022602083, -0.012697005, -0.013832112, 0.017284725, -0.0065014474, 0.047591258, 0.06283913, -0.032580387, -0.039906736, -0.050770573, 0.06505398, 0.004307632, -0.07331451, 0.03492824, 0.041894276, 0.039532218, -0.030327288, 0.0036885955, 0.016241407, 0.069296464, 0.07423016, 0.030480705, 0.064470015, 0.035091624, -0.050633598, -0.054948118, -0.007076625, 0.0050007687, -0.03635548, 0.07135583, -0.049124554, 0.0022741272, 0.065793045, -0.042549245, -0.0

In [18]:
from crewai_tools import WebsiteSearchTool
from crewai import Agent, LLM
from crewai.project import agent
from crewai_tools import RagTool
from crewai.tools import tool
from com.example.agentic.tools.config import _tool_config

#@tool
def website_search(http_url: str):
    """
    Useful for when you need to ask with lookup on website data.
    """
    #return retriever.get_relevant_documents()
    # Proper Configuration Structure
    search_tool = WebsiteSearchTool(
        website='https://example.com',
        config=_tool_config
    )

    return search_tool.run(http_url)

website_search("https://docs.crewai.com/how-to/Installing-CrewAI/")   

CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping URL validation for: https://example.com
CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping URL validation for: https://example.com


'Relevant Content:\n\nreduced production incidents through proactive defect detection. \n\n  \n\n• Developed the physical data model for Investment Product & Portfolio Governance data pipelines considering \n\ndifferent functional & operational risk reporting. Drive the modernization of IPGC & UPGC Data Platform by \n\nmoving to Cloud Foundry/AWS & Oracle Cloud Services \n\n \n\n• Led the strategic modernization of IPGC & UPGC Data Platform from on-premise to Cloud Foundry, AWS, and \n\nOracle Cloud Services, reducing infrastructure costs by an estimated 30–40% and improving platform scalability \n\nto support 3× data volume growth without linear headcount addition. \n\n \n\n• Established architecture governance practices through structured design review sessions with SMEs and cross-\n\nfunctional teams, standardizing data engineering patterns and reducing rework cycles by ~25%, accelerating \n\ntime-to-production for new pipeline features. \n\n \n\nSR. SOFTWARE ENGINEER | ACCENTURE SE

In [ ]:
researcher = Agent(
    llm=llm,
    name="Researcher",
    role="Researches topics by searching the website data.",
    tools=[website_search],
    goal="Answer questions by retrieving relevant information from the website's data.",  # Add the goal here
    backstory="You are a helpful AI assistant specializing in searching and retrieving information from a website. Use your 'Website_Search' tool to find relevant documents when answering questions."  
)

researcher_boss = Agent(
    llm=llm,
    name="Researcher Boss",
    role="Challenges the researcher to bring out the best out of his findings",
    tools=[website_search],
    goal="Ask further questions to the researcher and validates the retrieved relevant information from the website's data.",  # Add the goal here
    backstory="You are a helpful AI assistant boss and your job is to make sure the retrieved information is correct. Use your 'Website_Search' tool to find relevant documents when answering questions."  
)


In [ ]:
from crewai import Task
research_task = Task(
    description=(
        "Analyze the URL provided ({crewai_url}) "
        "to extract information about how crewai works. "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of crewai specifications, including necessary "
        "tools to get started"
    ),
    agent=researcher,
    #async_execution=True
)

boss_task = Task(
    description=(
        "Analyze the URL provided ({crewai_url}) "
        "to extract information about how crewai works. "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of crewai specifications, including necessary "
        "tools to get started"
    ),
    agent=researcher_boss,
    #async_execution=True
)


In [ ]:
from crewai import Crew
# Create Crew
research_crew1 = Crew(
    agents=[researcher, researcher_boss],
    tasks=[research_task, boss_task],
    verbose=True,  # This will print logs to the console as the crew works
    planning=True,
)

# Job Context
job_crew_works = {
    'crewai_url': 'https://docs.crewai.com/how-to/Installing-CrewAI/',
    'personal_writeup': """Accomplished Researcher 
    with 18 years of experience, specializing in
    setting up CrewAI kind of agent based systems"""
}

async_result = await research_crew1.kickoff_async(inputs=job_crew_works)
print(async_result)

# Kickoff the Crew's Work
#research_crew1.kickoff(inputs=job_crew_works)
#print(result)